In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Medical Study Retrieval — Topic & Outcome Search With Metadata

## 1. Project Overview

**Task:** Given a clinical question, retrieve the most relevant medical studies from an indexed corpus using both semantic search and structured metadata filters (study type, outcome, population, year).

**Approach:** Build a corpus of synthetic study abstracts → parse structured metadata → embed with sentence-transformers → retrieve with metadata-aware filtering → generate evidence summaries → evaluate retrieval quality.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for summarization
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store with metadata filtering

> **No API keys required.** Everything runs locally with Ollama.

---

## ⚠️ IMPORTANT LIMITATIONS — READ FIRST

**This notebook is a TECHNICAL DEMONSTRATION of retrieval techniques, NOT a medical tool.**

| Limitation | Explanation |
|-----------|-------------|
| **Not real studies** | All abstracts in this notebook are synthetic. They are NOT real published medical research. |
| **Not medical advice** | Nothing in this notebook should be used to make clinical decisions. |
| **No clinical validation** | The retrieval system has not been validated against real medical databases (PubMed, Cochrane, etc.). |
| **LLM hallucination risk** | The LLM may generate plausible-sounding but factually wrong medical statements. |
| **No expert review** | Real medical literature search requires expert librarians, systematic review methodology, and peer review. |
| **Bias in embeddings** | Embedding models are trained on general web text, not medical corpora. Medical-specific models (PubMedBERT, BioSentVec) would perform better in production. |

**For actual medical literature search, use:** PubMed, Cochrane Library, Embase, or consult a medical librarian.

---

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Metadata-aware retrieval** — filter by study type, outcome, population alongside semantic search |
| 2 | **Structured metadata extraction** — parse study design, sample size, outcome type from text |
| 3 | **Faceted search** — combine embedding similarity with structured field constraints |
| 4 | **Evidence summarization** — synthesize findings from multiple studies, citing each |
| 5 | **Retrieval evaluation** — Recall@k and MRR on ground-truth query-study pairs |
| 6 | **Responsible framing** — present retrieved evidence without medical overclaiming |

## 3. Problem Statement

Researchers and clinicians need to find relevant prior studies when:

- Reviewing existing evidence on a treatment
- Checking whether a specific outcome has been studied in a population
- Identifying gaps in the literature

Traditional keyword search on PubMed requires learning MeSH terms and Boolean logic. Semantic search can complement this by understanding queries in natural language — but it must be paired with structured metadata filters to be useful.

## 4. Why Metadata-Aware Retrieval

Pure semantic search is insufficient for medical literature because:

| Issue | Example |
|-------|--------|
| Ignores study design | You want RCTs, not case reports |
| Ignores population | You want studies in adults, not pediatric |
| Ignores outcome type | You want mortality data, not just symptom improvement |
| Ignores recency | Medical evidence evolves — a 2022 meta-analysis supersedes a 2015 pilot |

**Metadata-aware retrieval** = semantic similarity + structured filters on study metadata.

## 5. Pipeline Architecture

```
Study Abstracts (title + abstract + structured metadata)
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 1: CORPUS CONSTRUCTION                      │
  │  • 25 synthetic study abstracts                    │
  │  • Structured metadata: study_type, outcome,       │
  │    population, year, sample_size                   │
  └────────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 2: EMBED + INDEX                            │
  │  • Embed title + abstract with MiniLM              │
  │  • Store in ChromaDB with metadata fields          │
  └────────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 3: METADATA-AWARE RETRIEVAL                 │
  │  • Semantic query for topic                        │
  │  • Filter: study_type, outcome, population, year   │
  │  • Ranked by embedding similarity within filters   │
  └────────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 4: EVIDENCE SUMMARY                         │
  │  • Synthesize findings from retrieved studies       │
  │  • Cite each study; note limitations               │
  │  • Flag confidence level and evidence gaps          │
  └────────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 5: EVALUATION                               │
  │  • Recall@k and MRR on ground-truth pairs          │
  │  • Filtered vs unfiltered comparison               │
  └────────────────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

## 8. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5
TEMP_ANSWER = 0.1
TEMP_JUDGE = 0.0

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Retrieval top-k: {TOP_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Synthetic Study Corpus

## 10. Build the Study Dataset

**Reminder: All studies below are FICTIONAL.** They are designed to demonstrate retrieval techniques, not to represent actual medical evidence. Study IDs, authors, and findings are entirely synthetic.

Each study has structured metadata:
- **study_type** — RCT, cohort, meta_analysis, case_control, cross_sectional
- **outcome_type** — mortality, symptom_relief, biomarker, adverse_event, quality_of_life
- **population** — adults, elderly, pediatric, mixed
- **topic** — cardiovascular, diabetes, respiratory, mental_health, oncology
- **year** — publication year
- **sample_size** — number of participants

In [ ]:
STUDIES = [
    # --- Cardiovascular ---
    {"id": "S-001", "topic": "cardiovascular", "study_type": "RCT",
     "outcome_type": "mortality", "population": "elderly", "year": 2023, "sample_size": 2400,
     "title": "Intensive vs standard blood pressure targets in adults over 70",
     "abstract": "This randomized controlled trial compared intensive blood pressure lowering (target <120 mmHg systolic) with standard targets (<140 mmHg) in 2400 adults aged 70-85. Over a median follow-up of 3.2 years, the intensive group showed a 15% relative reduction in cardiovascular mortality (HR 0.85, 95% CI 0.74-0.98). However, the intensive group also had higher rates of hypotension-related falls (8.2% vs 3.1%). The trial suggests a net benefit for intensive targets but highlights the need to weigh fall risk in frail individuals."},

    {"id": "S-002", "topic": "cardiovascular", "study_type": "meta_analysis",
     "outcome_type": "mortality", "population": "adults", "year": 2024, "sample_size": 18500,
     "title": "Statin therapy and all-cause mortality: updated meta-analysis of 12 trials",
     "abstract": "This meta-analysis pooled data from 12 randomized trials (n=18,500) examining statin therapy for primary prevention. Statins were associated with a 12% reduction in all-cause mortality (RR 0.88, 95% CI 0.83-0.94, p<0.001) over a mean follow-up of 5 years. Subgroup analysis showed greater benefit in patients with elevated CRP. Heterogeneity was low (I²=18%). Limitations include variable statin types and doses across trials."},

    {"id": "S-003", "topic": "cardiovascular", "study_type": "cohort",
     "outcome_type": "biomarker", "population": "adults", "year": 2022, "sample_size": 5600,
     "title": "Mediterranean diet adherence and troponin levels in middle-aged adults",
     "abstract": "This prospective cohort followed 5600 adults aged 40-65 for 4 years. Higher adherence to the Mediterranean diet (top tertile vs bottom) was associated with 22% lower high-sensitivity troponin levels (p=0.003) and 18% lower NT-proBNP. Adjustment for BMI, smoking, and exercise attenuated the association modestly (to 17%). The study suggests dietary patterns may influence subclinical cardiac damage markers."},

    {"id": "S-004", "topic": "cardiovascular", "study_type": "RCT",
     "outcome_type": "adverse_event", "population": "adults", "year": 2023, "sample_size": 800,
     "title": "Myalgia incidence in high-dose vs moderate-dose statin regimens",
     "abstract": "This double-blind RCT randomized 800 adults with hyperlipidemia to high-dose (80mg) or moderate-dose (20mg) atorvastatin. After 12 months, myalgia was reported by 14.3% (high dose) vs 6.8% (moderate dose), p=0.001. CK elevation above 5x ULN occurred in 2.1% vs 0.5%. Most myalgia cases resolved within 4 weeks of dose reduction. The findings support starting at moderate doses and titrating based on tolerance."},

    {"id": "S-005", "topic": "cardiovascular", "study_type": "cross_sectional",
     "outcome_type": "biomarker", "population": "mixed", "year": 2021, "sample_size": 12000,
     "title": "Sleep duration and C-reactive protein: a population-level analysis",
     "abstract": "Cross-sectional analysis of 12,000 participants from a national health survey. Both short sleep (<6h) and long sleep (>9h) were associated with elevated CRP compared to 7-8h sleep (short: OR 1.34, 95% CI 1.18-1.52; long: OR 1.28, 95% CI 1.11-1.49). The U-shaped relationship persisted after adjusting for BMI, age, and comorbidities. Causal inference is limited by the cross-sectional design."},

    # --- Diabetes ---
    {"id": "S-006", "topic": "diabetes", "study_type": "RCT",
     "outcome_type": "biomarker", "population": "adults", "year": 2024, "sample_size": 1200,
     "title": "Continuous glucose monitoring vs finger-prick in type 2 diabetes management",
     "abstract": "This 6-month RCT randomized 1200 adults with type 2 diabetes (HbA1c 7.5-10%) to continuous glucose monitoring (CGM) or standard finger-prick testing. CGM users achieved a mean HbA1c reduction of 1.1% vs 0.6% for finger-prick (p<0.001). Time in range (70-180 mg/dL) improved by 12 percentage points with CGM. Hypoglycemic events were similar between groups. The study supports CGM for glycemic management beyond type 1 diabetes."},

    {"id": "S-007", "topic": "diabetes", "study_type": "cohort",
     "outcome_type": "mortality", "population": "elderly", "year": 2023, "sample_size": 8400,
     "title": "Metformin continuation after 80 years of age and survival outcomes",
     "abstract": "Retrospective cohort of 8400 patients with type 2 diabetes who reached age 80 while on metformin. Continued use (n=5200) was associated with lower all-cause mortality compared to discontinuation (n=3200) over 3 years (HR 0.78, 95% CI 0.69-0.88). Benefits persisted after adjustment for renal function.  Observational design limits causal claims; indication bias and unmeasured confounders are possible."},

    {"id": "S-008", "topic": "diabetes", "study_type": "meta_analysis",
     "outcome_type": "symptom_relief", "population": "adults", "year": 2022, "sample_size": 6800,
     "title": "Exercise interventions for diabetic neuropathy pain: systematic review",
     "abstract": "Systematic review and meta-analysis of 14 trials (n=6800) evaluating exercise for diabetic peripheral neuropathy pain. Structured exercise programs (aerobic + resistance, 3x/week, 12+ weeks) reduced pain scores by a standardized mean difference of -0.45 (95% CI -0.62 to -0.28, p<0.001). Balance training additionally reduced fall risk. Heterogeneity was moderate (I²=42%). Most trials were small and unblinded, limiting evidence quality."},

    {"id": "S-009", "topic": "diabetes", "study_type": "case_control",
     "outcome_type": "adverse_event", "population": "adults", "year": 2021, "sample_size": 3200,
     "title": "SGLT2 inhibitor use and risk of diabetic ketoacidosis: case-control study",
     "abstract": "Case-control study comparing 1600 cases of diabetic ketoacidosis (DKA) with 1600 matched controls among type 2 diabetes patients. SGLT2 inhibitor use was associated with increased DKA risk (OR 2.1, 95% CI 1.5-2.9). Euglycemic DKA (blood glucose <250 mg/dL) accounted for 38% of cases among SGLT2i users. Risk factors included concurrent illness, surgery, and carbohydrate restriction. Findings support targeted patient education on DKA recognition."},

    {"id": "S-010", "topic": "diabetes", "study_type": "RCT",
     "outcome_type": "quality_of_life", "population": "adults", "year": 2024, "sample_size": 450,
     "title": "Structured diabetes education program and patient-reported quality of life",
     "abstract": "RCT of 450 adults with newly diagnosed type 2 diabetes randomized to a 12-week structured education program or standard care. At 6 months, the education group reported significantly higher diabetes-specific quality of life (DQOL score: 72 vs 61, p<0.001) and greater self-efficacy. HbA1c improvements were modest (0.3% difference, p=0.04). The study was unblinded, which may have influenced self-reported outcomes."},

    # --- Respiratory ---
    {"id": "S-011", "topic": "respiratory", "study_type": "RCT",
     "outcome_type": "symptom_relief", "population": "adults", "year": 2023, "sample_size": 950,
     "title": "Inhaled corticosteroids vs combination therapy in moderate COPD",
     "abstract": "This 12-month RCT compared inhaled corticosteroid (ICS) monotherapy with ICS + long-acting beta-agonist (LABA) combination in 950 adults with moderate COPD. The combination group had 28% fewer moderate exacerbations (rate ratio 0.72, 95% CI 0.61-0.85) and improved FEV1 (+110 mL vs +40 mL). Pneumonia risk was similar between groups (3.2% vs 2.8%). Findings support combination therapy as first-line for moderate COPD."},

    {"id": "S-012", "topic": "respiratory", "study_type": "cohort",
     "outcome_type": "mortality", "population": "elderly", "year": 2022, "sample_size": 14000,
     "title": "Influenza vaccination and pneumonia mortality in adults over 65",
     "abstract": "Population-based cohort of 14,000 adults ≥65 followed over 5 influenza seasons. Annual vaccination was associated with 35% lower pneumonia-related mortality (adjusted HR 0.65, 95% CI 0.55-0.77) and 22% lower all-cause winter mortality. The healthy vaccinee effect may partially explain the results — healthier individuals are more likely to get vaccinated. Sensitivity analyses using instrumental variables narrowed the benefit estimate to 20-28%."},

    {"id": "S-013", "topic": "respiratory", "study_type": "meta_analysis",
     "outcome_type": "symptom_relief", "population": "pediatric", "year": 2024, "sample_size": 4200,
     "title": "Honey vs dextromethorphan for nighttime cough in children: meta-analysis",
     "abstract": "Meta-analysis of 8 RCTs (n=4200 children aged 2-12) comparing honey with dextromethorphan for acute nighttime cough. Honey showed comparable cough reduction (SMD -0.09, 95% CI -0.28 to 0.10, p=0.35) with fewer reported side effects. Honey was superior to no treatment (SMD -0.62, 95% CI -0.81 to -0.43). Limitations: variable honey types/doses, parental reporting bias, short follow-up (1-3 nights)."},

    {"id": "S-014", "topic": "respiratory", "study_type": "cross_sectional",
     "outcome_type": "biomarker", "population": "adults", "year": 2023, "sample_size": 7500,
     "title": "Air pollution exposure and spirometry values in urban vs rural populations",
     "abstract": "Cross-sectional study of 7500 adults comparing spirometry between high-pollution urban areas and rural controls. Urban residents showed 5.2% lower FEV1 and 4.8% lower FVC (both p<0.001) after adjusting for smoking, age, and occupation. PM2.5 exposure was the strongest predictor (beta -0.032L per 10µg/m³ increase). Cross-sectional design precludes causation. Healthy-worker effect may bias results."},

    # --- Mental Health ---
    {"id": "S-015", "topic": "mental_health", "study_type": "RCT",
     "outcome_type": "symptom_relief", "population": "adults", "year": 2024, "sample_size": 680,
     "title": "Cognitive behavioral therapy vs medication for moderate depression: 12-month outcomes",
     "abstract": "This 12-month RCT compared CBT (16 sessions) with sertraline (50-200mg) in 680 adults with moderate depression (PHQ-9 10-19). Both groups showed substantial improvement. At 12 months: CBT response rate 62%, sertraline 58% (difference not significant, p=0.31). CBT had lower relapse rates at 18-month follow-up (24% vs 39%, p=0.004). Side effects were more common with sertraline (41% vs 12% for CBT). The study was not blinded to treatment assignment."},

    {"id": "S-016", "topic": "mental_health", "study_type": "meta_analysis",
     "outcome_type": "symptom_relief", "population": "adults", "year": 2023, "sample_size": 11200,
     "title": "Exercise as adjunct therapy for anxiety disorders: meta-analysis of 18 trials",
     "abstract": "Meta-analysis of 18 RCTs (n=11,200) evaluating structured exercise as adjunct to standard treatment for generalized anxiety, panic, and social anxiety disorders. Exercise as adjunct reduced anxiety scores (SMD -0.52, 95% CI -0.68 to -0.36, p<0.001). Aerobic exercise (running, cycling) showed larger effects than resistance training. Effects attenuated when limited to blinded assessments. Publication bias was detected via funnel plot asymmetry."},

    {"id": "S-017", "topic": "mental_health", "study_type": "cohort",
     "outcome_type": "quality_of_life", "population": "adults", "year": 2022, "sample_size": 3800,
     "title": "Social isolation and quality of life trajectories over 5 years",
     "abstract": "Longitudinal cohort of 3800 adults followed for 5 years using validated quality of life instruments. Persistently socially isolated individuals showed a 2.3-point annual decline in SF-36 mental component scores (p<0.001) and 1.8-point decline in physical component scores. Onset of isolation was associated with steeper decline than chronic isolation. Residual confounding by depression severity is possible."},

    {"id": "S-018", "topic": "mental_health", "study_type": "RCT",
     "outcome_type": "adverse_event", "population": "elderly", "year": 2023, "sample_size": 1500,
     "title": "Benzodiazepine tapering with psychological support in older adults",
     "abstract": "RCT of 1500 adults aged ≥65 on long-term benzodiazepines randomized to supervised tapering with CBT support vs usual care. At 6 months, 42% of the tapering group had successfully discontinued vs 12% usual care (p<0.001). Withdrawal-related adverse events were managed through gradual dose reduction (10% every 2 weeks). Rebound insomnia occurred in 28% of the tapering group but resolved in most by 3 months. The study supports structured deprescribing."},

    {"id": "S-019", "topic": "mental_health", "study_type": "cross_sectional",
     "outcome_type": "biomarker", "population": "adults", "year": 2024, "sample_size": 2200,
     "title": "Screen time and cortisol patterns in working-age adults",
     "abstract": "Cross-sectional study measuring salivary cortisol (4 daily samples) in 2200 working adults and correlating with self-reported screen time. Screen time >6h/day was associated with a flattened diurnal cortisol slope (beta -0.18, p=0.002) and elevated evening cortisol. Reverse causation is possible: stressed individuals may engage in more screen time. Objective screen-time measurement was not available."},

    # --- Oncology ---
    {"id": "S-020", "topic": "oncology", "study_type": "RCT",
     "outcome_type": "mortality", "population": "adults", "year": 2024, "sample_size": 3100,
     "title": "Adjuvant immunotherapy vs chemotherapy in stage III non-small cell lung cancer",
     "abstract": "Phase III RCT comparing adjuvant immunotherapy (anti-PD-L1) with standard platinum-based chemotherapy in 3100 patients with resected stage III NSCLC. At 3-year follow-up, immunotherapy showed improved disease-free survival (DFS: 62% vs 48%, HR 0.68, 95% CI 0.59-0.78, p<0.001). Overall survival data are not yet mature. Grade 3-4 immune-related adverse events occurred in 18% of the immunotherapy group. Final OS analysis is planned at 5-year follow-up."},

    {"id": "S-021", "topic": "oncology", "study_type": "cohort",
     "outcome_type": "quality_of_life", "population": "adults", "year": 2023, "sample_size": 2600,
     "title": "Long-term quality of life after breast cancer treatment: 10-year follow-up",
     "abstract": "Prospective cohort of 2600 breast cancer survivors followed for 10 years post-treatment. At 10 years, 34% reported persistent fatigue and 28% reported cognitive concerns. Physical QoL returned to population norms by year 5 for most patients. Mental health QoL remained lower, particularly in those who received chemotherapy (mean SF-36 MCS: 44 vs 48 for surgery-only, p=0.008). Survivorship programs improved outcomes modestly."},

    {"id": "S-022", "topic": "oncology", "study_type": "meta_analysis",
     "outcome_type": "adverse_event", "population": "adults", "year": 2022, "sample_size": 9800,
     "title": "Immune checkpoint inhibitor-related thyroid dysfunction: pooled analysis",
     "abstract": "Pooled analysis of 22 trials (n=9800) of anti-PD-1/PD-L1 therapy. Thyroid dysfunction occurred in 12.4% of patients (hypothyroidism 9.1%, hyperthyroidism 3.3%). Combination immunotherapy doubled the risk (OR 2.2). Most cases were grade 1-2 and managed with hormone replacement. Severe thyroid storm occurred in 0.3%. Early routine thyroid monitoring is recommended. Across-trial variability in reporting limits precision of incidence estimates."},

    {"id": "S-023", "topic": "oncology", "study_type": "case_control",
     "outcome_type": "biomarker", "population": "adults", "year": 2023, "sample_size": 1800,
     "title": "Circulating tumor DNA as early recurrence marker in colorectal cancer",
     "abstract": "Case-control study of 900 colorectal cancer patients with post-surgical recurrence matched to 900 non-recurrent controls. Detectable ctDNA at 4 weeks post-surgery predicted recurrence with sensitivity 68% and specificity 92%. ctDNA-positive patients had a median lead time of 8.2 months before radiographic recurrence (95% CI 6.4-10.1). False negatives were more common in mucinous histology. The study supports ctDNA surveillance but clinical utility requires RCT validation."},

    {"id": "S-024", "topic": "oncology", "study_type": "RCT",
     "outcome_type": "symptom_relief", "population": "adults", "year": 2024, "sample_size": 420,
     "title": "Acupuncture for chemotherapy-induced nausea: sham-controlled trial",
     "abstract": "Double-blind sham-controlled RCT of 420 patients receiving highly emetogenic chemotherapy. True acupuncture vs sham acupuncture over 3 cycles. Complete nausea control: 48% (true) vs 37% (sham), p=0.03. The effect size was modest (NNT=9). Both groups were also receiving standard antiemetic therapy. Blinding assessment showed 60% of participants correctly guessed their group, potentially unblinding the trial. Results are preliminary and need to be replicated in larger samples."},

    {"id": "S-025", "topic": "oncology", "study_type": "cross_sectional",
     "outcome_type": "quality_of_life", "population": "mixed", "year": 2023, "sample_size": 4500,
     "title": "Financial toxicity and treatment adherence across cancer types",
     "abstract": "Cross-sectional survey of 4500 cancer patients across 12 cancer types. Financial toxicity (COST score <26) was reported by 31% of patients. Those with financial toxicity were 2.3x more likely to miss or delay treatment (OR 2.3, 95% CI 1.9-2.8). Younger patients (<50) and uninsured patients were most affected. Self-reported adherence may overestimate actual compliance. Geographic and insurance system variation limits generalizability."},
]

print(f"Total studies: {len(STUDIES)}")
print(f"\nTopics: {Counter(s['topic'] for s in STUDIES)}")
print(f"Study types: {Counter(s['study_type'] for s in STUDIES)}")
print(f"Outcomes: {Counter(s['outcome_type'] for s in STUDIES)}")
print(f"Populations: {Counter(s['population'] for s in STUDIES)}")
print(f"\nYear range: {min(s['year'] for s in STUDIES)}-{max(s['year'] for s in STUDIES)}")
print(f"Sample size range: {min(s['sample_size'] for s in STUDIES):,}-{max(s['sample_size'] for s in STUDIES):,}")

## 11. Inspect the Corpus

In [ ]:
print("STUDY CORPUS (all synthetic — not real medical evidence)")
print("=" * 100)
for s in STUDIES:
    type_short = {"RCT": "RCT", "meta_analysis": "M-A", "cohort": "COH",
                  "case_control": "C-C", "cross_sectional": "X-S"}.get(s["study_type"], "???")
    print(f"  {s['id']} [{type_short:3s}] [{s['topic']:14s}] [{s['outcome_type']:15s}] "
          f"n={s['sample_size']:>6,} {s['year']} | {s['title'][:48]}")

---

# Part B — Embedding & Indexing

## 12. Prepare and Embed Studies

In [ ]:
def study_to_text(study: dict) -> str:
    """Combine title and abstract for embedding."""
    return f"{study['title']}\n{study['abstract']}"


study_texts = [study_to_text(s) for s in STUDIES]
study_metadatas = [
    {
        "study_id": s["id"],
        "topic": s["topic"],
        "study_type": s["study_type"],
        "outcome_type": s["outcome_type"],
        "population": s["population"],
        "year": s["year"],
        "sample_size": s["sample_size"],
        "title": s["title"],
    }
    for s in STUDIES
]

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("med_studies")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="med_studies",
    metadata={"hnsw:space": "cosine"},
)

vectors = embeddings.embed_documents(study_texts)
collection.add(
    documents=study_texts,
    embeddings=vectors,
    metadatas=study_metadatas,
    ids=[s["id"] for s in STUDIES],
)

print(f"Indexed {collection.count()} studies in ChromaDB")
print(f"Embedding dim: {len(vectors[0])}")
print(f"Avg text length: {sum(len(t) for t in study_texts) // len(study_texts)} chars")

---

# Part C — Metadata-Aware Retrieval

## 13. Core Retrieval Function

The key idea: semantic similarity narrows by **topic relevance**, while metadata filters narrow by **study characteristics**. Together they answer queries like:

> *"Find RCTs on blood pressure in elderly patients published after 2022 that measured mortality."*

In [ ]:
def retrieve_studies(query: str, top_k: int = TOP_K,
                     study_type: Optional[str] = None,
                     outcome_type: Optional[str] = None,
                     population: Optional[str] = None,
                     topic: Optional[str] = None,
                     min_year: Optional[int] = None) -> list[dict]:
    """Retrieve studies with optional metadata filters."""
    query_vector = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_vector], "n_results": top_k}

    # Build ChromaDB where filter
    conditions = []
    if study_type:
        conditions.append({"study_type": study_type})
    if outcome_type:
        conditions.append({"outcome_type": outcome_type})
    if population:
        conditions.append({"population": population})
    if topic:
        conditions.append({"topic": topic})
    if min_year:
        conditions.append({"year": {"$gte": min_year}})

    if len(conditions) == 1:
        kwargs["where"] = conditions[0]
    elif len(conditions) > 1:
        kwargs["where"] = {"$and": conditions}

    results = collection.query(**kwargs)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
            "similarity": 1 - results["distances"][0][i],
        })
    return hits


def display_studies(hits: list[dict], label: str = ""):
    if label:
        print(f"\n  [{label}]")
    for i, h in enumerate(hits, 1):
        m = h["metadata"]
        type_short = {"RCT": "RCT", "meta_analysis": "M-A", "cohort": "COH",
                      "case_control": "C-C", "cross_sectional": "X-S"}.get(m["study_type"], "???")
        print(f"    {i}. sim={h['similarity']:.3f} | {m['study_id']} [{type_short}] "
              f"{m['year']} n={m['sample_size']:,} | {m['title'][:50]}")


# Test: basic semantic retrieval
q = "statin therapy and cardiovascular outcomes"
print(f"Q: {q}")
hits = retrieve_studies(q, top_k=3)
display_studies(hits, "No filters")

## 14. Demonstrate Metadata Filters

In [ ]:
q = "blood pressure treatment in older patients"
print(f"Q: {q}")

print("\n--- No filters ---")
display_studies(retrieve_studies(q, top_k=3), "All studies")

print("\n--- Filter: RCTs only ---")
display_studies(retrieve_studies(q, top_k=3, study_type="RCT"), "RCTs")

print("\n--- Filter: Mortality outcomes only ---")
display_studies(retrieve_studies(q, top_k=3, outcome_type="mortality"), "Mortality")

print("\n--- Filter: Elderly population + RCT ---")
display_studies(retrieve_studies(q, top_k=3, study_type="RCT", population="elderly"), "Elderly RCTs")

In [ ]:
q = "depression treatment effectiveness"
print(f"Q: {q}")

print("\n--- No filters ---")
display_studies(retrieve_studies(q, top_k=3), "All")

print("\n--- Filter: mental_health + RCT + 2023+ ---")
display_studies(
    retrieve_studies(q, top_k=3, topic="mental_health", study_type="RCT", min_year=2023),
    "Mental health RCTs 2023+"
)

In [ ]:
q = "side effects of cancer immunotherapy"
print(f"Q: {q}")

print("\n--- Filter: oncology + adverse_event ---")
display_studies(
    retrieve_studies(q, top_k=3, topic="oncology", outcome_type="adverse_event"),
    "Oncology adverse events"
)

print("\n--- Filter: oncology + meta_analysis ---")
display_studies(
    retrieve_studies(q, top_k=3, topic="oncology", study_type="meta_analysis"),
    "Oncology meta-analyses"
)

---

# Part D — Evidence Summary Generation

## 15. Evidence Summary System Prompt

The system prompt enforces responsible evidence presentation — no overclaiming, mandatory limitation disclosure, and explicit caveat that these are synthetic studies.

In [ ]:
EVIDENCE_SYSTEM = """You are a research evidence summarizer. You summarize retrieved study findings.

CRITICAL RULES:
1. These are SYNTHETIC studies created for demonstration. State this at the top.
2. NEVER present findings as medical advice or treatment recommendations.
3. Cite every claim as [Study: S-XXX].
4. Report effect sizes, confidence intervals, and sample sizes when available.
5. Always note study limitations explicitly.
6. Use hedging language: 'suggests', 'was associated with', 'in this study' — never 'proves' or 'shows definitively'.
7. Note the evidence level: RCT > cohort > case-control > cross-sectional > case report.
8. If studies disagree, present both sides.
9. End with an 'Evidence Gaps' section noting what is NOT covered.
10. End with: 'This summary is for educational purposes only. Consult healthcare professionals for medical decisions.'"""


def format_study_sources(hits: list[dict]) -> str:
    parts = []
    for h in hits:
        m = h["metadata"]
        # Retrieve the full abstract from the original dataset
        full_study = next((s for s in STUDIES if s["id"] == m["study_id"]), None)
        abstract = full_study["abstract"] if full_study else h["text"]
        parts.append(
            f"[Study: {m['study_id']}] {m['title']}\n"
            f"Type: {m['study_type']} | Outcome: {m['outcome_type']} | "
            f"Pop: {m['population']} | n={m['sample_size']:,} | Year: {m['year']}\n"
            f"{abstract}"
        )
    return "\n\n".join(parts)


def summarize_evidence(query: str, **filter_kwargs) -> dict:
    """Full pipeline: retrieve studies -> generate evidence summary."""
    hits = retrieve_studies(query, **filter_kwargs)
    sources = format_study_sources(hits)

    prompt = (
        f"Summarize the evidence for this clinical question:\n\n"
        f"QUESTION: {query}\n\n"
        f"RETRIEVED STUDIES:\n{sources}\n\n"
        "Return ONLY JSON:\n"
        '{"summary": "evidence summary with [Study: S-XXX] citations and limitation notes",'
        ' "evidence_level": "strong|moderate|weak|insufficient",'
        ' "studies_cited": ["S-XXX"],'
        ' "key_findings": ["finding 1", "finding 2"],'
        ' "limitations": ["limitation 1", "limitation 2"],'
        ' "evidence_gaps": ["gap 1"],'
        ' "disclaimer": "standard medical disclaimer"}'
    )

    resp = ask(prompt, system=EVIDENCE_SYSTEM, temperature=TEMP_ANSWER)
    result = parse_json(resp)
    if not result:
        result = {
            "summary": resp,
            "evidence_level": "unknown",
            "studies_cited": [h["metadata"]["study_id"] for h in hits],
            "key_findings": [],
            "limitations": ["LLM did not return valid JSON"],
            "evidence_gaps": [],
            "disclaimer": "This is for educational purposes only.",
        }
    result["hits"] = hits
    return result


def display_evidence(query: str, result: dict):
    level = result.get("evidence_level", "?")
    badge = {"strong": "STRONG", "moderate": "MOD", "weak": "WEAK",
             "insufficient": "INSUF"}.get(level, "???")

    print(f"\n{'='*80}")
    print(f"Q: {query}")
    print(f"Evidence level: {badge}")
    print(f"{'='*80}")
    print(f"\nSUMMARY:")
    wrap_print(result.get("summary", ""))

    if result.get("key_findings"):
        print(f"\nKEY FINDINGS:")
        for f in result["key_findings"][:5]:
            print(f"  • {textwrap.shorten(str(f), width=90, placeholder='...')}")

    if result.get("limitations"):
        print(f"\nLIMITATIONS:")
        for lim in result["limitations"][:5]:
            print(f"  ⚠ {textwrap.shorten(str(lim), width=90, placeholder='...')}")

    if result.get("evidence_gaps"):
        print(f"\nEVIDENCE GAPS:")
        for gap in result["evidence_gaps"][:3]:
            print(f"  ? {textwrap.shorten(str(gap), width=90, placeholder='...')}")

    print(f"\n  Studies cited: {', '.join(result.get('studies_cited', []))}")
    print(f"\n  ⚕ {result.get('disclaimer', 'For educational purposes only.')}")


print("Evidence summary pipeline ready")

## 16. Generate Evidence Summaries

In [ ]:
q1 = "What does the evidence say about statin therapy for reducing mortality?"
r1 = summarize_evidence(q1, topic="cardiovascular", outcome_type="mortality")
display_evidence(q1, r1)

In [ ]:
q2 = "How effective is CBT compared to medication for depression?"
r2 = summarize_evidence(q2, topic="mental_health", study_type="RCT")
display_evidence(q2, r2)

In [ ]:
q3 = "What are the side effects of immunotherapy in cancer patients?"
r3 = summarize_evidence(q3, topic="oncology", outcome_type="adverse_event")
display_evidence(q3, r3)

In [ ]:
q4 = "Is continuous glucose monitoring better than finger-prick testing for diabetes?"
r4 = summarize_evidence(q4, topic="diabetes")
display_evidence(q4, r4)

## 17. Multi-Filter Complex Query

In [ ]:
q5 = "RCTs published since 2023 measuring symptom relief in respiratory conditions"
r5 = summarize_evidence(
    q5,
    study_type="RCT",
    outcome_type="symptom_relief",
    topic="respiratory",
    min_year=2023,
)
display_evidence(q5, r5)

## 18. Broad Query Across Topics

In [ ]:
q6 = "What meta-analyses are available on treatment effectiveness?"
r6 = summarize_evidence(q6, study_type="meta_analysis")
display_evidence(q6, r6)

## 19. Filtered vs Unfiltered Comparison

This shows why metadata filters matter — the same query gets very different results depending on constraints.

In [ ]:
q = "exercise and health outcomes"
print(f"Q: {q}")

print("\n--- UNFILTERED (top 5) ---")
unfiltered = retrieve_studies(q, top_k=5)
display_studies(unfiltered, "No filters")

print("\n--- FILTERED: mental_health only ---")
filtered_mh = retrieve_studies(q, top_k=3, topic="mental_health")
display_studies(filtered_mh, "Mental health")

print("\n--- FILTERED: diabetes + symptom_relief ---")
filtered_dm = retrieve_studies(q, top_k=3, topic="diabetes", outcome_type="symptom_relief")
display_studies(filtered_dm, "Diabetes symptom relief")

print("\n  -> Same query, completely different relevant studies via metadata filters.")

---

# Part E — Retrieval Evaluation

## 20. Ground-Truth Evaluation Set

In [ ]:
EVAL_SET = [
    {"query": "intensive blood pressure lowering in elderly",
     "expected_ids": ["S-001"],
     "filters": {"study_type": "RCT", "population": "elderly"}},
    {"query": "do statins reduce mortality in primary prevention?",
     "expected_ids": ["S-002"],
     "filters": {"outcome_type": "mortality"}},
    {"query": "Mediterranean diet and heart damage markers",
     "expected_ids": ["S-003"],
     "filters": {"outcome_type": "biomarker"}},
    {"query": "statin side effects muscle pain",
     "expected_ids": ["S-004"],
     "filters": {"outcome_type": "adverse_event"}},
    {"query": "continuous glucose monitoring vs finger prick in type 2 diabetes",
     "expected_ids": ["S-006"],
     "filters": {"topic": "diabetes", "study_type": "RCT"}},
    {"query": "metformin use in very old patients and survival",
     "expected_ids": ["S-007"],
     "filters": {"population": "elderly"}},
    {"query": "exercise for diabetic nerve pain",
     "expected_ids": ["S-008"],
     "filters": {"topic": "diabetes", "outcome_type": "symptom_relief"}},
    {"query": "SGLT2 inhibitor and ketoacidosis risk",
     "expected_ids": ["S-009"],
     "filters": {"topic": "diabetes", "outcome_type": "adverse_event"}},
    {"query": "COPD inhaler combination therapy",
     "expected_ids": ["S-011"],
     "filters": {"topic": "respiratory"}},
    {"query": "flu vaccine and pneumonia death in elderly",
     "expected_ids": ["S-012"],
     "filters": {"topic": "respiratory", "population": "elderly"}},
    {"query": "CBT vs antidepressant for depression",
     "expected_ids": ["S-015"],
     "filters": {"topic": "mental_health", "study_type": "RCT"}},
    {"query": "exercise as treatment for anxiety",
     "expected_ids": ["S-016"],
     "filters": {"topic": "mental_health"}},
    {"query": "immunotherapy vs chemotherapy for lung cancer",
     "expected_ids": ["S-020"],
     "filters": {"topic": "oncology", "study_type": "RCT"}},
    {"query": "thyroid problems from checkpoint inhibitor drugs",
     "expected_ids": ["S-022"],
     "filters": {"topic": "oncology", "outcome_type": "adverse_event"}},
]

print(f"Evaluation set: {len(EVAL_SET)} query-study pairs")

## 21. Evaluate: Filtered vs Unfiltered Retrieval

In [ ]:
def run_evaluation(eval_set: list[dict], use_filters: bool, top_k: int = 5) -> dict:
    """Compute Recall@k and MRR."""
    hits_at_k = 0
    reciprocal_ranks = []
    details = []

    for item in eval_set:
        filter_kwargs = item["filters"] if use_filters else {}
        hits = retrieve_studies(item["query"], top_k=top_k, **filter_kwargs)
        retrieved_ids = [h["metadata"]["study_id"] for h in hits]
        expected = item["expected_ids"]

        found_rank = None
        for rank, rid in enumerate(retrieved_ids, 1):
            if rid in expected:
                found_rank = rank
                break

        if found_rank is not None:
            hits_at_k += 1
            reciprocal_ranks.append(1.0 / found_rank)
        else:
            reciprocal_ranks.append(0.0)

        details.append({
            "query": item["query"],
            "expected": expected,
            "top_3": retrieved_ids[:3],
            "hit": found_rank is not None,
            "rank": found_rank,
        })

    n = len(eval_set)
    return {
        "recall_at_k": hits_at_k / n,
        "mrr": sum(reciprocal_ranks) / n,
        "n": n,
        "k": top_k,
        "details": details,
    }


eval_unfiltered = run_evaluation(EVAL_SET, use_filters=False, top_k=5)
eval_filtered = run_evaluation(EVAL_SET, use_filters=True, top_k=5)

print("RETRIEVAL EVALUATION: FILTERED vs UNFILTERED")
print("=" * 60)
print(f"{'Metric':<20} {'Unfiltered':>12} {'Filtered':>12}")
print("-" * 60)
print(f"{'Recall@5':<20} {eval_unfiltered['recall_at_k']:>11.1%} {eval_filtered['recall_at_k']:>11.1%}")
print(f"{'MRR':<20} {eval_unfiltered['mrr']:>11.3f} {eval_filtered['mrr']:>11.3f}")
print(f"{'Questions':<20} {eval_unfiltered['n']:>12} {eval_filtered['n']:>12}")

In [ ]:
print("\nPER-QUERY COMPARISON")
print("=" * 90)

filt_wins = 0
unfilt_wins = 0
ties = 0

for ud, fd in zip(eval_unfiltered["details"], eval_filtered["details"]):
    u_rank = ud["rank"] or 999
    f_rank = fd["rank"] or 999

    if f_rank < u_rank:
        winner = "FILT"
        filt_wins += 1
    elif u_rank < f_rank:
        winner = "UNFL"
        unfilt_wins += 1
    else:
        winner = "TIE "
        ties += 1

    u_s = f"@{ud['rank']}" if ud["hit"] else "MISS"
    f_s = f"@{fd['rank']}" if fd["hit"] else "MISS"
    print(f"  [{winner}] Unfilt={u_s:>5} Filt={f_s:>5} | {ud['query'][:55]}")

print(f"\nFiltered wins: {filt_wins} | Unfiltered wins: {unfilt_wins} | Ties: {ties}")

## 22. Evaluation at Different k

In [ ]:
print("RECALL AT DIFFERENT K")
print("=" * 55)
print(f"{'k':>3} | {'Unfilt R@k':>12} {'Unfilt MRR':>12} | {'Filt R@k':>12} {'Filt MRR':>12}")
print("-" * 55)
for k in [1, 3, 5]:
    u = run_evaluation(EVAL_SET, use_filters=False, top_k=k)
    f = run_evaluation(EVAL_SET, use_filters=True, top_k=k)
    print(f"  {k:>1} | {u['recall_at_k']:>11.1%} {u['mrr']:>11.3f} | {f['recall_at_k']:>11.1%} {f['mrr']:>11.3f}")

## 23. Summary Quality Evaluation (LLM-as-Judge)

In [ ]:
JUDGE_SYSTEM = "You evaluate medical evidence summaries for quality and responsible presentation."

JUDGE_PROMPT = """Evaluate this evidence summary.

QUESTION: {question}
SUMMARY: {summary}
STUDIES CITED: {citations}

Score each dimension 1-5:
- accuracy: does the summary faithfully represent the studies?
- no_overclaiming: does it avoid presenting findings as definitive medical advice?
- limitations_noted: are study limitations properly disclosed?
- citations: are study references present and relevant?
- evidence_gaps: does it acknowledge what evidence is missing?

Return ONLY JSON:
{{"accuracy": N, "no_overclaiming": N, "limitations_noted": N,
  "citations": N, "evidence_gaps": N,
  "overall": N, "notes": "brief explanation"}}"""

judge_samples = [
    (q1, r1),
    (q2, r2),
    (q3, r3),
]

print("EVIDENCE SUMMARY QUALITY (LLM-as-Judge)")
print("=" * 80)

for q, result in judge_samples:
    citations = ", ".join(result.get("studies_cited", [])[:4])
    resp = ask(
        JUDGE_PROMPT.format(
            question=q,
            summary=result.get("summary", "")[:500],
            citations=citations,
        ),
        system=JUDGE_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(resp)
    if scores:
        print(f"\n  Q: {q[:60]}")
        for dim in ["accuracy", "no_overclaiming", "limitations_noted", "citations", "evidence_gaps"]:
            val = scores.get(dim, "?")
            bar = "*" * int(val) if isinstance(val, (int, float)) else "?"
            print(f"    {dim:20s}: {val}/5 {bar}")
        print(f"    {'overall':20s}: {scores.get('overall', '?')}/5")
        if scores.get("notes"):
            print(f"    Notes: {scores['notes'][:80]}")

---

## 24. Limitations of This Approach

This section is not optional. Anyone using retrieval for medical literature **must** understand:

### Fundamental Limitations

| Limitation | Consequence | Mitigation |
|-----------|-------------|------------|
| **Synthetic corpus** | None of these studies are real. No real evidence was retrieved. | Use PubMed, Cochrane, Embase for real searches |
| **General-purpose embeddings** | `all-MiniLM-L6-v2` was not trained on medical text; may miss biomedical synonyms | Use PubMedBERT, BioSentVec, or SciSpaCy for production |
| **Small corpus (n=25)** | Real medical databases have tens of millions of abstracts. Retrieval at scale requires different architecture. | Test with actual PubMed subsets |
| **No MESH/ontology** | Medical search relies on controlled vocabularies (MeSH, SNOMED) for precision. We use none. | Integrate UMLS/MeSH-based query expansion |
| **LLM hallucination** | The LLM may fabricate statistics, study details, or medical claims. Every claim must be verified against the source abstract. | Always cite; never trust uncited claims |
| **No GRADE assessment** | Real evidence quality uses GRADE (Grading of Recommendations, Assessment, Development, and Evaluation). We use a simplified heuristic. | Implement GRADE-based evidence grading |
| **Publication bias** | Real literature has publication bias (positive results are more likely to be published). Our synthetic corpus has no such analysis. | Include funnel plot analysis, pre-registration checks |
| **No full-text** | We only index abstracts. Full-text methods sections, supplementary data, and tables contain crucial detail. | Index full-text when available |

### What This System Is NOT

- ❌ **NOT** a diagnostic tool
- ❌ **NOT** a treatment recommendation engine
- ❌ **NOT** a replacement for systematic review methodology
- ❌ **NOT** a substitute for expert medical librarians
- ❌ **NOT** validated for clinical decision-making

### What This System IS

- ✅ A **technical demonstration** of metadata-aware retrieval
- ✅ A **learning tool** for understanding semantic search + structured filters
- ✅ A **starting point** for building a real medical search system (which requires extensive validation)

## 25. Common Mistakes

| Mistake | Why It's Bad | Better Approach |
|---------|-------------|----------------|
| Treating LLM summaries as evidence | LLMs hallucinate; summaries may contain fabricated claims | Always verify against the source abstract |
| Ignoring study design in retrieval | An observational study does not have the same weight as an RCT | Filter by study_type and note evidence hierarchy |
| No limitation disclosure | Users may overinterpret results | Always list limitations explicitly |
| Using general embeddings for medical text | Misses domain-specific synonyms and relationships | Use biomedical embedding models in production |
| Not filtering by population | Adult evidence may not apply to pediatric populations | Always include population in query and filters |
| Presenting p-values without context | p=0.04 is not 'strong proof' | Report effect sizes, confidence intervals, and sample sizes |

## 26. Production Improvement Ideas

1. **PubMed integration** — index real abstracts via the PubMed E-utilities API
2. **Biomedical embeddings** — use PubMedBERT or BioSentVec instead of general MiniLM
3. **MeSH query expansion** — map user queries to MeSH terms for precise retrieval
4. **GRADE evidence grading** — implement standardized evidence quality assessment
5. **Citation graph** — show which studies cite each other and their relationship
6. **Full-text indexing** — index methods and results sections, not just abstracts
7. **Living review mode** — automatically re-run queries monthly to detect new evidence

## 27. Exercises

### Exercise 1
Add 5 new studies to a new topic (e.g., "infectious_disease"). Write evaluation pairs and measure if retrieval quality holds. How does cross-topic retrieval change?

### Exercise 2
Implement a `study_type_hierarchy` scorer: retrieve all matching studies but weight RCTs and meta-analyses higher than cohort and cross-sectional studies in the result ranking.

### Exercise 3
Build a citation verifier: after generating an evidence summary, extract all `[Study: S-XXX]` citations and check each against the actual retrieved studies. Report the percentage of verified vs hallucinated citations.

### Mini Challenge
Implement a PICO (Population, Intervention, Comparison, Outcome) query parser: take a natural language question and use the LLM to extract P, I, C, O components, then map each to metadata filters for more precise retrieval.

In [ ]:
print("SESSION SUMMARY")
print("=" * 60)
print(f"Studies indexed: {len(STUDIES)} (ALL SYNTHETIC)")
print(f"Topics: {len(set(s['topic'] for s in STUDIES))}")
print(f"Study types: {len(set(s['study_type'] for s in STUDIES))}")
print(f"Evaluation pairs: {len(EVAL_SET)}")
print(f"\nRetrieval results (Recall@5 / MRR):")
print(f"  Unfiltered: {eval_unfiltered['recall_at_k']:.1%} / {eval_unfiltered['mrr']:.3f}")
print(f"  Filtered:   {eval_filtered['recall_at_k']:.1%} / {eval_filtered['mrr']:.3f}")
print(f"\nCapabilities built:")
print(f"  - Metadata-aware retrieval (topic, study_type, outcome, population, year)")
print(f"  - Evidence summarization with mandatory limitation disclosure")
print(f"  - Filtered vs unfiltered comparison")
print(f"  - Retrieval evaluation (Recall@k, MRR)")
print(f"  - LLM-as-judge with medical-specific dimensions")
print(f"\n⚠ ALL STUDIES ARE SYNTHETIC. NOT FOR MEDICAL DECISIONS.")
print(f"⚕ For real medical evidence, use PubMed, Cochrane, or consult a professional.")

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Metadata filters + semantic search** are more powerful than either alone |
| 2 | Medical retrieval MUST filter by **study type, population, outcome** — not just topic |
| 3 | **General embeddings are insufficient** for medical text — use biomedical models in production |
| 4 | **Never overclaim** — use hedging language, cite sources, list limitations |
| 5 | **Limitations sections are mandatory** — not optional, not brief, not vague |
| 6 | **Study design hierarchy matters** — RCT > cohort > case-control > cross-sectional |
| 7 | **LLMs hallucinate medical facts** — every claim must be verified against the source |
| 8 | **Evaluation sets** with ground-truth query-study pairs are essential |
| 9 | **PICO-structured queries** improve retrieval precision for clinical questions |
| 10 | This demo is for **learning retrieval techniques only** — not for clinical use |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*

**⚕ Disclaimer:** All studies in this notebook are synthetic. This is a technical demonstration of metadata-aware retrieval, not a medical evidence review tool. For medical decisions, consult healthcare professionals and use validated databases such as PubMed and Cochrane Library.